In [ ]:
import math
import pandas as pd

# Settings
file_path = '../../raw_data/LCI_SOL.xlsx'  # input Excel file
input_sheets = ["ssp1", "ssp2", "ssp5"]    # sheets to process
output_file = "../../generated_data/LCI_SOL_SSPs.xlsx"  # output Excel file
#output_file_normalized = "../../generated_data/LCI_SOL_SSPs_norm.xlsx" # output Excel file
output_file_expanded_norm = "../../generated_data/SSPs_SOL.xlsx"

scenario_names = ["BAU", "LC3", "EST"]     # columns
variable_bases = ["C_SOL", "M_SOL", "N_SOL"]  # row name bases

# Exponential-offset fit function
def fit_exponential_offset(E1, E2, E3, y1=2025, y2=2050, y3=2100):
    """
    Solve for parameters A, C, k of:
        E(t) = C + A * exp(k * (t - y1))
    that pass exactly through E(y1), E(y2), E(y3).
    """

    R = (E3 - E1) / (E2 - E1)
    y = (-1 + math.sqrt(4 * R - 3)) / 2
    k = math.log(y) / (y2 - y1)
    A = (E2 - E1) / (y - 1)
    C = E1 - A

    return A, C, k

# Process each sheet
output_dfs = {}

for sheet_name in input_sheets:
    # Read sheet, use first column (row labels) as index
    df = pd.read_excel(file_path, sheet_name=sheet_name, index_col=0)

    results = {}  # dict: row_label -> {year: value}

    for scenario in scenario_names:
        for var_base in variable_bases:
            row_label = f"{scenario}_{var_base}"  # e.g. BAU_C_SOL

            # Row names in the sheet, e.g. "C_SOL_2025"
            row_2025 = f"{var_base}_2025"
            row_2050 = f"{var_base}_2050"
            row_2100 = f"{var_base}_2100"

            # Extract values for this scenario & variable
            E2025 = df.loc[row_2025, scenario]
            E2050 = df.loc[row_2050, scenario]
            E2100 = df.loc[row_2100, scenario]

            # Fit parameters
            A, C, k = fit_exponential_offset(E2025, E2050, E2100)

            # Build yearly series 2025–2100
            yearly_values = {
                year: C + A * math.exp(k * (year - 2025))
                for year in range(2025, 2101)
            }

            results[row_label] = yearly_values

    # Convert results to DataFrame:
    # - rows: scenario_variable (e.g. BAU_C_SOL)
    # - columns: years 2025–2100
    years = list(range(2025, 2101))
    final_df = pd.DataFrame.from_dict(results, orient="index", columns=years)

    # Store for export
    output_dfs[sheet_name] = final_df

# Export to Excel
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    for sheet_name, out_df in output_dfs.items():
        out_df.to_excel(writer, sheet_name=sheet_name, index=True)


In [ ]:
normalized_dfs = {}

for sheet_name, out_df in output_dfs.items():
    # Divide each row by its 2025 value
    # out_df columns: 2025..2100, index: BAU_C_SOL, ...
    norm_df = out_df.div(out_df[2025], axis=0)
    normalized_dfs[sheet_name] = norm_df

# with pd.ExcelWriter(output_file_normalized, engine="openpyxl") as writer:
#     for sheet_name, norm_df in normalized_dfs.items():
#         norm_df.to_excel(writer, sheet_name=sheet_name, index=True)

substeps_per_year = 200
start_year = 2025
end_year = 2100

n_intervals = end_year - start_year          # 75 intervals (2025→2026, ..., 2099→2100)
total_points = n_intervals * substeps_per_year  # 75 * 200 = 15000

years = list(range(start_year, end_year + 1))  # 2025..2100 (76 values)

expanded_normalized_dfs = {}

for sheet_name, norm_df in normalized_dfs.items():
    expanded_data = {}

    for row_label in norm_df.index:
        # normalized yearly values (2025..2100)
        vals = norm_df.loc[row_label, years].values.astype(float)

        expanded = [0.0] * total_points

        # For each year interval
        for i in range(n_intervals):  # 0..74
            v0 = vals[i]       # value at year start
            v1 = vals[i + 1]   # value at next year

            for s in range(substeps_per_year):  # 0..199
                frac = s / (substeps_per_year - 1)  # 0 at start, 1 at s=199
                idx = i * substeps_per_year + s     # global index 0..14999
                expanded[idx] = v0 + frac * (v1 - v0)

        expanded_data[row_label] = expanded

    # rows: BAU_C_SOL, ..., EST_N_SOL
    # columns: 0..14999
    expanded_df = pd.DataFrame.from_dict(
        expanded_data,
        orient="index",
        columns=range(total_points)
    )

    expanded_normalized_dfs[sheet_name] = expanded_df
    
# Export expanded normalized data (0..14999) to Excel
with pd.ExcelWriter(output_file_expanded_norm, engine="openpyxl") as writer:
    for sheet_name, exp_df in expanded_normalized_dfs.items():
        exp_df.to_excel(writer, sheet_name=sheet_name, index=True)
